# Quick Start Guide - Azure AI Foundry

This notebook provides a hands-on introduction to Azure AI Foundry. You'll learn how to:
1. Initialize the AI Project client
2. List available models
3. Create a simple chat completion request
4. Create a basic AI agent
5. Handle basic error scenarios

## Prerequisites
- Completed environment setup from previous notebook
- Azure credentials configured

## Import Required Libraries and Setup

In the next cell, we'll:
1. Import the necessary Azure SDK libraries for authentication and AI Projects
2. Import standard Python libraries for environment variables and JSON handling
3. Initialize Azure credentials using DefaultAzureCredential
   - This will automatically use your logged-in Azure CLI credentials
   - Alternatively, it can use other authentication methods like environment variables or managed identity


In [1]:
# Import required libraries
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
import os
import json

# Initialize credentials
credential = DefaultAzureCredential()

## Creating the AI Project Client

In the next cell, we'll create an AI Project client using the connection string from our `.env` file.
> **Note:** This example uses the synchronous client. For higher performance scenarios, you can also create an asynchronous client by importing `asyncio` and using the async methods from `AIProjectClient`.

The client will be used to:
- Connect to your Azure AI Project using the connection string
- Authenticate using Azure credentials
- Enable making inference requests to your deployed models


In [ ]:
from dotenv import load_dotenv
from pathlib import Path
from urllib.parse import urlparse
import os
import requests

# Load environment variables from workspace root .env
notebook_path = Path().absolute()
load_dotenv(notebook_path.parent / '.env')

# Parse PROJECT_ENDPOINT into required AIProjectClient constructor components
_url            = os.getenv("PROJECT_ENDPOINT")
_parsed         = urlparse(_url)
base_endpoint   = f"{_parsed.scheme}://{_parsed.netloc}"
path_parts      = [p for p in _parsed.path.split("/") if p]
project_name    = os.getenv("AZURE_AI_PROJECT_NAME") or (path_parts[-1] if path_parts else "")
hub_name        = _parsed.netloc.split(".")[0]

# Auto-detect subscription_id & resource_group from Foundry hub
print("Auto-detecting subscription ID and resource group from Foundry hub...")
try:
    mgmt_token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {mgmt_token}"}

    # List all accessible subscriptions
    subs = requests.get(
        "https://management.azure.com/subscriptions?api-version=2020-01-01",
        headers=headers, timeout=15
    ).json().get("value", [])

    subscription_id = None
    resource_group = None
    
    for sub in subs:
        sub_id = sub["subscriptionId"]
        # Search for the Foundry hub: type=Microsoft.CognitiveServices/accounts, name=hub_name
        resources = requests.get(
            f"https://management.azure.com/subscriptions/{sub_id}/resources"
            f"?$filter=name eq '{hub_name}' and "
            f"resourceType eq 'Microsoft.CognitiveServices/accounts'"
            f"&api-version=2021-04-01",
            headers=headers, timeout=15
        ).json().get("value", [])

        if resources:
            # ARM resource ID: /subscriptions/<sub>/resourceGroups/<rg>/providers/...
            rg_from_id = resources[0]["id"].split("/")[4]
            subscription_id = sub_id
            resource_group = rg_from_id
            break

    if not (subscription_id and resource_group):
        raise RuntimeError(f"Hub '{hub_name}' not found in any accessible subscription.")

    print(f"✓ Subscription ID:  {subscription_id[:8]}...")
    print(f"✓ Resource group:   {resource_group}")

except Exception as e:
    raise EnvironmentError(
        f"Failed to auto-detect subscription and resource group: {e}"
    ) from e

try:
    client = AIProjectClient(
        endpoint=base_endpoint,
        subscription_id=subscription_id,
        resource_group_name=resource_group,
        project_name=project_name,
        credential=credential,
    )
    print("✓ Successfully initialized AIProjectClient")
except Exception as e:
    print(f"× Error initializing client: {str(e)}")

## Create a Simple Completion
Let's try a basic completion request:

Now that we have an authenticated client, let's use it to make a chat completion request.
The code below demonstrates how to:
1. Get a ChatCompletionsClient from the azure-ai-inference package
2. Use it to make a simple completion request

We'll use the MODEL_DEPLOYMENT_NAME from our `.env` file, making it easy to switch between different
deployed models without changing code. This could be an Azure OpenAI model, Microsoft model, or other providers
that support chat completions.

> Note: Make sure you have the azure-ai-inference package installed (from requirements.txt or as mentioned in [README.md](../README.md#-quick-start))


In [ ]:
# Chat completion via the new Azure AI Foundry portal
# Use OpenAI Python SDK with Foundry's OpenAI endpoint and API key

from openai import OpenAI

model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
openai_key = os.getenv("AZURE_OPENAI_KEY")

try:
    # Create OpenAI client using Foundry endpoint and API key
    openai_client = OpenAI(
        api_key=openai_key,
        base_url=openai_endpoint,
    )
    
    # Use the standard OpenAI Python SDK API for chat completions
    response = openai_client.chat.completions.create(
        model=model_deployment_name,
        messages=[
            {"role": "user", "content": "How to be healthy in one sentence?"}
        ],
    )
    
    print(response.choices[0].message.content)
except Exception as e:
    print(f"An error occurred: {str(e)}")

## Create a simple Agent

Using AI Agent Service, we can create a simple agent to answer health related questions.

Let's explore Azure AI Agent Service, a powerful tool for building intelligent agents.

Azure AI Agent Service is a fully managed service that helps developers build, deploy, and scale AI agents
without managing infrastructure. It combines large language models with tools that allow agents to:
- Answer questions using RAG (Retrieval Augmented Generation)
- Perform actions through tool calling 
- Automate complex workflows

The code below demonstrates how to:
1. Create an agent with a code interpreter tool
2. Create a conversation thread
3. Send a message requesting BMI analysis 
4. Process the request and get results
5. Save any generated visualizations

The agent will use the model specified in our .env file (MODEL_DEPLOYMENT_NAME) and will have access
to a code interpreter tool for creating visualizations. This showcases how agents can combine
natural language understanding with computational capabilities.

> The visualization will be saved as a PNG file in the same folder as this notebook.
 



In [ ]:
# BMI Analysis with Visualization
# Uses chat completion to generate code, then executes it locally

from openai import OpenAI
from pathlib import Path
import matplotlib.pyplot as plt

print("=== BMI Analysis with Visualization ===\n")

try:
    # Create OpenAI client
    openai_client = OpenAI(
        api_key=os.getenv("AZURE_OPENAI_KEY"),
        base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    )
    
    # Ask the model to generate Python code for BMI calculation and visualization
    print("Generating BMI analysis code...")
    response = openai_client.chat.completions.create(
        model=model_deployment_name,
        messages=[
            {
                "role": "user",
                "content": (
                    "Generate Python code to:\n"
                    "1. Calculate BMI for a 130 lb, 5'4\" (64 inches) female\n"
                    "2. Create a matplotlib bar chart showing BMI categories (Underweight <18.5, Normal 18.5-24.9, Overweight 25-29.9, Obese >=30)\n"
                    "3. Mark where the calculated BMI falls on the scale\n"
                    "4. Save as 'bmi_analysis.png' with tight layout\n"
                    "5. Print the BMI value and analysis\n\n"
                    "Return ONLY the Python code, no explanations."
                )
            }
        ],
    )
    
    code = response.choices[0].message.content
    print("Executing analysis code...\n")
    
    # Execute the generated code locally
    exec(code)
    
    # Verify the chart was saved
    output_file = Path.cwd() / "bmi_analysis.png"
    if output_file.exists():
        print(f"\n✓ Chart saved to: {output_file}")
        
except Exception as e:
    print(f"Error: {str(e)}")
    
    # Fallback: Generate the visualization directly
    print("\nGenerating fallback visualization...\n")
    
    try:
        # Calculate BMI manually
        weight_lbs = 130
        height_inches = 64
        bmi = (weight_lbs / (height_inches ** 2)) * 703
        
        print(f"BMI = {bmi:.1f} (Normal)")
        print(f"Analysis: A BMI of {bmi:.1f} falls in the Normal weight range on the standard scale (18.5-24.9).\n")
        
        # Create visualization
        fig, ax = plt.subplots(figsize=(12, 4))
        
        # BMI categories
        categories = ['Underweight', 'Normal', 'Overweight', 'Obese']
        ranges = [(0, 18.5), (18.5, 24.9), (25, 29.9), (30, 35)]
        colors = ['#5B9BD5', '#70AD47', '#FFC000', '#FF6B6B']
        
        # Draw category bars
        for i, (category, (start, end), color) in enumerate(zip(categories, ranges, colors)):
            ax.barh(0, end - start, left=start, height=0.5, color=color, edgecolor='black', linewidth=2)
            ax.text((start + end) / 2, 0, category, ha='center', va='center', fontweight='bold', fontsize=10, color='white')
        
        # Mark the calculated BMI
        ax.plot(bmi, 0, 'k|', markersize=20, markeredgewidth=3)
        ax.text(bmi, 0.35, f'BMI {bmi:.1f}\n(Normal)', ha='center', fontsize=11, fontweight='bold')
        
        # Labels and title
        ax.set_ylim(-1, 1)
        ax.set_xlim(15, 35)
        ax.set_xlabel('BMI', fontsize=12, fontweight='bold')
        ax.set_title('BMI Analysis — 130 lbs, 64 in', fontsize=14, fontweight='bold')
        ax.set_yticks([])
        
        # Save and display
        output_path = Path.cwd() / "bmi_analysis.png"
        fig.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        
        print(f"✓ Chart saved to: {output_path}")
        
    except Exception as fallback_error:
        print(f"Fallback error: {str(fallback_error)}")